In [1]:
# 1. IMPORTS
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.model_selection import cross_val_score

In [2]:
# 2. CHARGEMENT DES DONNÉES
train_test_dir = "../data/train_test"

X_train = pd.read_csv(f"{train_test_dir}/X_train.csv")
X_test  = pd.read_csv(f"{train_test_dir}/X_test.csv")
# squeeze() transforme un DataFrame d'une colonne en une simple "Series" (vecteur)
y_train = pd.read_csv(f"{train_test_dir}/y_train.csv").squeeze()
y_test  = pd.read_csv(f"{train_test_dir}/y_test.csv").squeeze()

print(f"Dimensions X_train : {X_train.shape}")
print(f"Dimensions X_test  : {X_test.shape}")

Dimensions X_train : (3497, 69)
Dimensions X_test  : (875, 69)


In [3]:
# 3. CRÉATION ET ENTRAÎNEMENT DU MODÈLE (Le "Fit")
model = RandomForestClassifier(
    n_estimators=200,         # Le modèle va planter 200 arbres de décision différents
    max_depth=10,             # Chaque arbre aura une profondeur maximale de 10 (évite le surapprentissage)
    class_weight="balanced",  # TRÈS IMPORTANT : Gère le déséquilibre (s'il y a peu de gens qui churnent)
    random_state=42,          # Garantit que tu auras le même résultat à chaque exécution
    n_jobs=-1                 # Utilise tous les cœurs de ton processeur pour aller plus vite
)

print("Entraînement en cours...")
model.fit(X_train, y_train)
print("Entraînement terminé !")

Entraînement en cours...
Entraînement terminé !


In [4]:
# 4. ÉVALUATION
# Le modèle donne sa réponse catégorique (0 ou 1)
y_pred = model.predict(X_test)

# Le modèle donne son degré de certitude (ex: 85% de chances de churn)
y_proba = model.predict_proba(X_test)[:, 1] 

print("Rapport de Classification :\n")
print(classification_report(y_test, y_pred, target_names=["Fidèle", "Churné"]))

print(f"Score ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}")
print(f"Score F1 (Churn) : {f1_score(y_test, y_pred):.4f}")
# Affiche l'importance des variables
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(5))

Rapport de Classification :

              precision    recall  f1-score   support

      Fidèle       0.93      0.95      0.94       584
      Churné       0.90      0.86      0.88       291

    accuracy                           0.92       875
   macro avg       0.92      0.91      0.91       875
weighted avg       0.92      0.92      0.92       875

Score ROC-AUC : 0.9691
Score F1 (Churn) : 0.8803
FavoriteSeason_Automne    0.194403
PreferredMonth            0.140471
LoyaltyLevel              0.073021
MonetaryTotal             0.040745
Frequency                 0.039518
dtype: float64


In [5]:
# 1. LISTER LES COLONNES TRICHEUSES
# (errors='ignore' évite que le code plante si tu as déjà supprimé les colonnes)
cols_to_drop = ['Recency', 'ChurnRiskCategory']

# 2. NETTOYER LES DONNÉES
X_train_clean = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test_clean = X_test.drop(columns=cols_to_drop, errors='ignore')

print(f"Anciennes dimensions : {X_train.shape[1]} colonnes.")
print(f"Nouvelles dimensions : {X_train_clean.shape[1]} colonnes.\n")

# 3. RÉ-ENTRAÎNER LE MODÈLE
# On réinitialise un cerveau tout neuf
model_realiste = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Entraînement du modèle sans triche en cours...")
model_realiste.fit(X_train_clean, y_train)

# 4. NOUVELLE ÉVALUATION
y_pred_clean = model_realiste.predict(X_test_clean)
y_proba_clean = model_realiste.predict_proba(X_test_clean)[:, 1]

print("\n--- RÉSULTATS RÉALISTES (SANS DATA LEAKAGE) ---")
print(classification_report(y_test, y_pred_clean, target_names=["Fidèle", "Churné"]))
print(f"Nouveau Score ROC-AUC : {roc_auc_score(y_test, y_proba_clean):.4f}")

Anciennes dimensions : 69 colonnes.
Nouvelles dimensions : 69 colonnes.

Entraînement du modèle sans triche en cours...

--- RÉSULTATS RÉALISTES (SANS DATA LEAKAGE) ---
              precision    recall  f1-score   support

      Fidèle       0.93      0.95      0.94       584
      Churné       0.90      0.86      0.88       291

    accuracy                           0.92       875
   macro avg       0.92      0.91      0.91       875
weighted avg       0.92      0.92      0.92       875

Nouveau Score ROC-AUC : 0.9691


In [6]:
# On mène l'enquête sur le nouveau modèle
importances_clean = pd.Series(model_realiste.feature_importances_, index=X_train_clean.columns)

print(" Les nouvelles colonnes suspectes :")
print(importances_clean.sort_values(ascending=False).head(10))

 Les nouvelles colonnes suspectes :
FavoriteSeason_Automne     0.194403
PreferredMonth             0.140471
LoyaltyLevel               0.073021
MonetaryTotal              0.040745
Frequency                  0.039518
TotalTransactions          0.037725
AvgDaysBetweenPurchases    0.036228
TotalQuantity              0.035994
UniqueInvoices             0.034843
UniqueDescriptions         0.029211
dtype: float64


In [7]:
# 1. LA LISTE COMPLÈTE DES COLONNES TRICHEUSES
cols_to_drop = [
    'Recency', 
    'ChurnRiskCategory', 
    'CustomerType_Perdu',
    'CustomerType_Occasionnel',
    'CustomerType_Nouveau',
    'RFMSegment'
]

# 2. NETTOYER LES DONNÉES
X_train_clean = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test_clean = X_test.drop(columns=cols_to_drop, errors='ignore')

print(f"Nouvelles dimensions : {X_train_clean.shape[1]} colonnes.\n")

# 3. RÉ-ENTRAÎNER LE MODÈLE DÉFINITIF
model_realiste = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model_realiste.fit(X_train_clean, y_train)

# 4. ÉVALUATION FINALE
y_pred_clean = model_realiste.predict(X_test_clean)
y_proba_clean = model_realiste.predict_proba(X_test_clean)[:, 1]

print("--- RÉSULTATS DÉFINITIFS (SANS TRICHE) ---")
print(classification_report(y_test, y_pred_clean, target_names=["Fidèle", "Churné"]))
print(f"Score ROC-AUC final : {roc_auc_score(y_test, y_proba_clean):.4f}")

Nouvelles dimensions : 69 colonnes.

--- RÉSULTATS DÉFINITIFS (SANS TRICHE) ---
              precision    recall  f1-score   support

      Fidèle       0.93      0.95      0.94       584
      Churné       0.90      0.86      0.88       291

    accuracy                           0.92       875
   macro avg       0.92      0.91      0.91       875
weighted avg       0.92      0.92      0.92       875

Score ROC-AUC final : 0.9691


In [8]:
# 1. LA LISTE NOIRE DÉFINITIVE
cols_to_drop = [
    'Recency', 
    'ChurnRiskCategory', 
    'CustomerType_Perdu',
    'CustomerType_Occasionnel',
    'CustomerType_Nouveau',
    'RFMSegment',
    # Les suspects temporels indirects :
    'MonetaryPerDay',
    'TenureRatio',
    'CustomerTenureDays',
    'FirstPurchaseDaysAgo'
]

# 2. NETTOYER LES DONNÉES
X_train_clean = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test_clean = X_test.drop(columns=cols_to_drop, errors='ignore')

# 3. RÉ-ENTRAÎNER LE MODÈLE DÉFINITIF
model_realiste = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1
)
model_realiste.fit(X_train_clean, y_train)

# 4. ÉVALUATION FINALE
y_pred_clean = model_realiste.predict(X_test_clean)
y_proba_clean = model_realiste.predict_proba(X_test_clean)[:, 1]

print("--- RÉSULTATS DÉFINITIFS (PURGE TOTALE) ---")
print(classification_report(y_test, y_pred_clean, target_names=["Fidèle", "Churné"]))
print(f"Score ROC-AUC final : {roc_auc_score(y_test, y_proba_clean):.4f}")

--- RÉSULTATS DÉFINITIFS (PURGE TOTALE) ---
              precision    recall  f1-score   support

      Fidèle       0.93      0.95      0.94       584
      Churné       0.90      0.86      0.88       291

    accuracy                           0.92       875
   macro avg       0.92      0.91      0.91       875
weighted avg       0.92      0.92      0.92       875

Score ROC-AUC final : 0.9691
